In [0]:

columns = ["month","quater","half_yearly"]

calendar = [
    (1,1,1),
    (2,1,1),
    (3,1,1),
    (4,2,1),
    (5,2,1),
    (6,2,1),
    (7,3,2),
    (8,3,2),
    (9,3,2),
    (10,4,2),
    (11,4,2),
    (12,4,2)
]
calendar_df = spark.createDataFrame(calendar,columns)

calendar_df.write.format('delta').mode('overwrite').saveAsTable('gold_medical_data.azure_blob_storage.dim_calendar')

In [0]:
encounters_df = spark.read.table("silver_medical_data.azure_blob_storage.encounters").alias("e")
payers_df = spark.read.table("silver_medical_data.azure_blob_storage.payers").alias("p")

dim_encounters_df = encounters_df.join(
    payers_df, encounters_df.payer_id == payers_df.id, "inner"
).selectExpr(
    "e.id AS encounter_id",
    "e.patient_id",
    "e.payer_id",
    "e.organization_id",
    "e.code AS encounter_code",
    "e.reason_code",
    "e.encounter_class",
    "e.start AS encounter_start",
    "e.stop AS encounter_stop",
    "p.name AS payer_name",
    "p.address AS payer_address",
    "p.city AS payer_city",
    "p.state_headquartered AS payer_state",
    "p.zip AS payer_zip"
)

dim_encounters_df.write.format("delta").mode("overwrite").saveAsTable(
    "gold_medical_data.azure_blob_storage.dim_encounters"
)

In [0]:
procedures_df = spark.read.table('silver_medical_data.azure_blob_storage.procedures')
dim_procedures_df = procedures_df.selectExpr(
    'patient_id',
    'encounter_id',
    'code as procedure_code',
    'description as procedure_code_description',
    'reason_code',
    'start as procedure_start',
    'stop as procedure_stop'
)

dim_procedures_df.write.format('delta').mode('overwrite').saveAsTable('gold_medical_data.azure_blob_storage.dim_procedures')